# 01 — Conform: every source into one table

**Goal:** take five files with five different layouts and turn them into **one table with
the same columns**.

**Why:** the matcher compares every record with every other record. Each file names its
columns differently:

| File | Name column | Street column |
|---|---|---|
| ERP | `customer_name` | `address_line_1` |
| Salesforce | `account_name` | `billing_street` |
| Alpha | `ship_to_name` | `ship_to_address` |
| Beta | `CustomerName` | `Street` |

If every file kept its own layout, we'd need separate matching code for ERP-vs-Salesforce,
ERP-vs-Alpha, Salesforce-vs-Beta… ten combinations. Put them all in one shape, and the
matching code is written **once**.

The real work happens in `src/mdm/conform.py`. This notebook just **uses** it and shows the result.

## Setup

**What the code does**

- `from mdm.conform import one_table, mapping` — brings in two things from `conform.py`:
  - `one_table` — the function that turns one file into the common shape
  - `mapping` — the dictionary that says, for each file, which column holds what
- `load(name)` — the same shortcut as in the profiling notebook.
  `dtype=str` keeps every column as text, so ZIP codes keep their leading zeros.

Both `one_table` **and** `mapping` must be imported. Importing only `one_table` gives the
error *"missing 1 required positional argument: 'mapping'"* — the function needs the
dictionary passed in, and the notebook doesn't know about it unless you import it.

In [5]:
import pandas as pd
from pathlib import Path
from mdm.conform import one_table, mapping

path = Path("../data/sample")

def load(name):
    return pd.read_csv(path / f"{name}.csv", dtype=str)

## What the mapping looks like

**Why:** `mapping` is the instruction sheet. Before running anything, it helps to see one entry.

**What the code does:** shows the entry for Beta — the most unusual file.

**How to read it:** the left side is *what we want*, the right side is *where Beta keeps it*.

- `"name_raw": "CustomerName"` → Beta's name is in the column called `CustomerName`
- `"suite_std": ["Street", "CustomerName"]` → a **list**: the suite can be in the street
  **or** typed into the name (D-012), so look in both
- `"zip5": ""` → Beta has **no** ZIP column of its own…
- `"city_state_zip": "CityStateZip"` → …it's hidden inside `"Chicago, IL 60611"`

**Adding a new distributor = adding one entry like this.** No new code.

In [6]:
mapping["beta"]

{'record_uid': 'RecordID',
 'source_system': 'beta',
 'role': 'tracing',
 'name_raw': 'CustomerName',
 'street_std': 'Street',
 'suite_std': ['Street', 'CustomerName'],
 'zip5': '',
 'city_state_zip': 'CityStateZip',
 'legacy_customer_id': '',
 'npi': ''}

## Build the combined table

**What the code does, line by line**

| Line | Meaning |
|---|---|
| `files = [...]` | a list of pairs: *(short name, file name)* |
| `for name, file in files:` | take each pair in turn |
| `load(file)` | read that CSV |
| `one_table(..., name, mapping)` | turn it into the common shape |
| `tables.append(...)` | add the result to a list |
| `pd.concat(tables, ignore_index=True)` | stack all the tables on top of each other |

`ignore_index=True` numbers the rows 0, 1, 2… across the whole result, instead of each file
starting again at 0.

**Gamma is left out on purpose.** It's the April file — it's used later to test what happens
when a *new* file arrives (A.10).

### What `one_table` produces — the common shape

| Column | What it holds | Example | Why |
|---|---|---|---|
| `record_uid` | unique ID across **all** files | `erp::E1001` | `E1001` could exist in two files; `erp::E1001` can't |
| `source_system` | which file | `erp` | |
| `source_record_id` | the file's own ID | `E1001` | to trace back to the original |
| `role` | kind of source | `master` / `crm` / `reference` / `tracing` | a tracing record matching nothing means something different from an ERP record matching nothing |
| `name_raw` | the name, untouched | `ST JOSEPH HOSP` | when a match goes wrong, you need to see what arrived |
| `name_std` | the cleaned name | `SAINT JOSEPH HOSPITAL` | D-009, D-010, D-012, D-013 |
| `street_std` | the cleaned street, no suite | `1240 MAIN STREET` | |
| `suite_std` | just the suite | `400` | found wherever it hides — D-003, D-004, D-012 |
| `zip5` | 5-digit ZIP | `97205` | ZIP+4 split off — D-005 |
| `legacy_customer_id` | the ERP's own customer number | `L-0001` | empty for files that don't have one |
| `npi` | the official healthcare ID | `1003811509` | only Salesforce and reference have it — D-008 |

**What to look for:** 406 rows — erp 63, sf 62, reference 61, alpha 156, beta 64.

In [7]:
files = [
    ("erp",               "erp_customer_master"),
    ("sf",                "salesforce_accounts"),
    ("reference",         "provider_reference"),
    ("distributor_alpha", "distributor_alpha_tracing_2026-03"),
    ("distributor_beta",  "distributor_beta_tracing_2026-03"),
]

tables = []
for name, file in files:
    tables.append(one_table(load(file), name, mapping))

records = pd.concat(tables, ignore_index=True)
records

,record_uid,source_system,source_record_id,role,name_raw,name_std,street_std,suite_std,zip5,legacy_customer_id,npi
0,erp::E1001,erp,E1001,master,OAKMONT IMAGING CTR,OAKMONT IMAGING CENTER,505 ELM STREET,400,97205,L-0001,
1,erp::E1002,erp,E1002,master,LAKEVIEW ORTHO,LAKEVIEW ORTHOPEDICS,505 ELM STREET,210,97205,L-0002,
2,erp::E1003,erp,E1003,master,NORTHGATE REGIONAL HOSP,NORTHGATE REGIONAL HOSPITAL,88 CEDAR LANE,,97205,L-0003,
3,erp::E1004,erp,E1004,master,BROOKSIDE SURGERY CTR,BROOKSIDE SURGERY CENTER,300 BIRCHWOOD COURT,,97205,L-0004,
4,erp::E1005,erp,E1005,master,ST LUKE COMMUNITY HOSP,SAINT LUKE COMMUNITY HOSPITAL,505 PINE AVENUE,,97205,L-0005,
...,...,...,...,...,...,...,...,...,...,...,...
401,distributor_beta::B-77060,distributor_beta,B-77060,tracing,Oakmont Imaging Ctr Ste 400,OAKMONT IMAGING CENTER,505 ELM STREET,400,97205,,
402,distributor_beta::B-77061,distributor_beta,B-77061,tracing,Peña Blanca Family Clinic,PENA BLANCA FAMILY CLINIC,900 BIRCHWOOD COURT,,,,
403,distributor_beta::B-77062,distributor_beta,B-77062,tracing,Peña Blanca Family Clnc,PENA BLANCA FAMILY CLINIC,900 BIRCHWOOD COURT,,32801,,
404,distributor_beta::B-77063,distributor_beta,B-77063,tracing,"Willamette Regional Hospital, Inc.",WILLAMETTE REGIONAL HOSPITAL,1240 HARBOR BOULEVARD,,61602,,


## Before and after

**Why:** to see the cleaning working on real rows from every file.

**What the code does**
- `records[[...]]` — the double brackets pick **several** columns
- `.sample(15)` — 15 **random** rows, so you see a mix of files. Run it a few times.

**What to look for:** messy names on the left (`ST JOSEPH HOSP`, `Saint Joseph Hospital`)
becoming the same clean name on the right (`SAINT JOSEPH HOSPITAL`).

In [8]:
records[["source_system", "name_raw", "name_std", "suite_std", "zip5"]].sample(15)

,source_system,name_raw,name_std,suite_std,zip5
279,distributor_alpha,PEÑA BLANCA FAMILY CLNC,PENA BLANCA FAMILY CLINIC,,32801
350,distributor_beta,Saint Joseph Medical Center Inc,SAINT JOSEPH MEDICAL CENTER,,80202
204,distributor_alpha,LAKEVIEW HOSP,LAKEVIEW HOSPITAL,,97205
15,erp,BROOKSIDE OUTPATIENT CTR,BROOKSIDE OUTPATIENT CENTER,2B,60611
46,erp,RIVERBEND CANCER INST,RIVERBEND CANCER INSTITUTE,,80202
69,sf,Mercy Outpatient Center,MERCY OUTPATIENT CENTER,120,97205
251,distributor_alpha,WILLAMETTE REGIONAL HOSPITAL D/B/A WILLAMETTE ...,WILLAMETTE REGIONAL HOSPITAL,,61602
54,erp,SILVER CREEK FAMILY CLNC,SILVER CREEK FAMILY CLINIC,,50309
45,erp,LAKEVIEW COMMUNITY HOSP,LAKEVIEW COMMUNITY HOSPITAL,,80202
159,reference,Northgate Surgery Center,NORTHGATE SURGERY CENTER,,33602


## A preview of matching

**Why:** before writing any matching code, a quick test of the idea — if two records have
the same cleaned name **and** the same ZIP, are they the same customer?

**What the code does**

| Line | Meaning |
|---|---|
| `records["name_zip"] = name_std + "\|" + zip5` | glue name and ZIP into one key: `SAINT JOSEPH HOSPITAL\|97205` |
| `.groupby("name_zip")` | put records with the same key together |
| `["source_system"].nunique()` | count how many **different files** are in each group |
| `.sort_values(ascending=False).head(10)` | show the 10 groups spanning the most files |

The `|` is just a separator, so the name and ZIP can't run into each other.

**What to look for:** 406 records fall into **93** groups, and **64** of those groups hold
records from two or more files. ERP, Salesforce, reference and the distributors — all
landing on the same key.

**That is matching**, done with nothing fancier than cleaning plus grouping. Step A.3 does it
properly.

In [9]:
records["name_zip"] = records["name_std"] + "|" + records["zip5"]
records.groupby("name_zip")["source_system"].nunique().sort_values(ascending=False).head(10)

name_zip
BAYVIEW ORTHOPEDICS|97401                5
BROOKSIDE CANCER INSTITUTE|80202         5
FAIRHAVEN HOSPITAL|55401                 5
CEDAR RIDGE IMAGING CENTER|50309         5
CEDAR RIDGE CANCER INSTITUTE|32801       5
FAIRHAVEN CANCER INSTITUTE|61602         5
GRANITE FALL COMMUNITY HOSPITAL|61602    5
SAINT LUKE COMMUNITY HOSPITAL|97205      5
SAINT MARY ORTHOPEDICS|32801             5
SAINT MARY CANCER INSTITUTE|33602        5
Name: source_system, dtype: int64

## …but name + ZIP alone is not enough

**Why:** a check on the preview above — does a group really hold *one* customer?
`SUMMIT REGIONAL HOSPITAL|61602` is the group with the **most records** (11), so it's the
one worth checking first.

**What the code does:** keeps only the rows whose key is `SUMMIT REGIONAL HOSPITAL|61602`,
and shows which street each one is on.

`records[ condition ]` keeps only the rows where the condition is True.

**What to look for:** these records sit on **two different streets** — `900 SUMMIT DRIVE`
and `1802 GRANT PARKWAY`. They are the two Summit Regional campuses found during profiling
(D-006): same name, same ZIP, **two different customers**, with two different legacy IDs in
the ERP.

So name + ZIP wrongly lumps them together. That's why the matching step also compares the
**street** — and why matching is more than grouping on one key.

In [10]:
summit = records[records["name_zip"] == "SUMMIT REGIONAL HOSPITAL|61602"]
summit[["source_system", "name_raw", "street_std", "legacy_customer_id"]]

,source_system,name_raw,street_std,legacy_customer_id
25,erp,SUMMIT REGIONAL HOSP,900 SUMMIT DRIVE,L-0026
26,erp,SUMMIT REGIONAL HOSP,1802 GRANT PARKWAY,L-0027
88,sf,Summit Regional Hospital,900 SUMMIT DRIVE,
89,sf,Summit Regional Hospital,1802 GRANT PARKWAY,
150,reference,Summit Regional Hospital,900 SUMMIT DRIVE,
151,reference,Summit Regional Hospital,1802 GRANT PARKWAY,
252,distributor_alpha,SUMMIT REGIONAL HOSPITAL,900 SUMMIT DRIVE,
253,distributor_alpha,SUMMIT REGIONAL HOSPITAL,1802 GRANT PARKWAY,
254,distributor_alpha,SUMMIT REGIONAL HOSP,1802 GRANT PARKWAY,
256,distributor_alpha,SUMMIT REGIONAL HOSPITAL,1802 GRANT PARKWAY,


## Summary

- Five files, five layouts → **one table, 406 rows, one shape**
- Every `record_uid` is unique across all files
- Cleaned names already line up across systems: 64 name+ZIP groups span two or more files
- **But** the same name and ZIP can still be two different customers (Summit Regional) —
  so the street has to be part of matching

**Next:** A.3 — deterministic matching. Find records that are *definitely* the same
customer because they share an exact value: the same NPI, the same legacy ID, or the same
cleaned name **and** address.